# 06 · Scikit-learn Pipeline
One pipeline from **raw** `application_train.csv` rows to a default score.

In [1]:
import sys
sys.path.insert(0, "..")

import json
import hashlib
import joblib
import pandas as pd
import numpy as np
from sklearn.model_selection import train_test_split
from sklearn.compose import ColumnTransformer
from sklearn.pipeline import Pipeline
from sklearn.impute import SimpleImputer
from sklearn.preprocessing import StandardScaler, OneHotEncoder, FunctionTransformer
from sklearn.ensemble import HistGradientBoostingClassifier

from src.features import add_application_features   # sentinel fix + the 8 engineered features
from src.metrics import evaluate

In [2]:
df = pd.read_csv("../data/application_train.csv")
X = df.drop(columns=["TARGET"])
y = df["TARGET"]
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, stratify=y, random_state=42)

# 1. Build the pipeline
`SK_ID_CURR` (an ID) and `CODE_GENDER` (not allowed) are not in the column lists, so the `ColumnTransformer` drops them.

In [3]:
features = add_application_features(X_train.drop(columns=["SK_ID_CURR", "CODE_GENDER"]))
numeric_cols = features.select_dtypes(include="number").columns.tolist()
categorical_cols = features.select_dtypes(exclude="number").columns.tolist()

preprocessor = ColumnTransformer([
    ("numeric", Pipeline([
        ("imputer", SimpleImputer(strategy="median", add_indicator=True)),
        ("scaler", StandardScaler()),
    ]), numeric_cols),
    ("categorical", Pipeline([
        ("imputer", SimpleImputer(strategy="most_frequent")),
        ("encoder", OneHotEncoder(handle_unknown="ignore", sparse_output=False)),
    ]), categorical_cols),
])

best_params = json.load(open("../reports/tuned_params.json"))   # from notebook 04
credit_pipeline = Pipeline([
    ("features", FunctionTransformer(add_application_features)),
    ("preprocessor", preprocessor),
    ("model", HistGradientBoostingClassifier(class_weight="balanced", early_stopping=False,
                                             random_state=42, **best_params)),
])

# 2. Fit on raw data and evaluate

In [4]:
credit_pipeline.fit(X_train, y_train)
y_prob = credit_pipeline.predict_proba(X_test)[:, 1]
evaluate(y_test, y_prob).round(4).to_frame("Pipeline (raw CSV in)")

,Pipeline (raw CSV in)
AUROC,0.7679
Gini,0.5357
KS,0.4021
AUPRC,0.2566
F1,0.3186
F1_threshold,0.6516


**AUROC 0.768 from raw CSV rows**, above the 0.72 target. It is 0.003 below the research model (0.771) because the pipeline only receives the application row, not the `bureau.csv` aggregates (those would need a lookup at scoring time).

# 3. Save and test on 5 raw rows

In [5]:
joblib.dump(credit_pipeline, "../models/credit_scoring_pipeline.pkl")

loaded = joblib.load("../models/credit_scoring_pipeline.pkl")
sample = pd.read_csv("../data/application_train.csv", nrows=5).drop(columns="TARGET")
loaded.predict_proba(sample)[:, 1].round(4)

array([0.8672, 0.3068, 0.2463, 0.3487, 0.4762])

# 4. Hash the data and the model

In [6]:
def sha256(path):
    h = hashlib.sha256()
    with open(path, "rb") as f:
        for chunk in iter(lambda: f.read(1 << 20), b""):
            h.update(chunk)
    return h.hexdigest()

hashes = pd.Series({
    "data/application_train.csv": sha256("../data/application_train.csv"),
    "models/credit_scoring_pipeline.pkl": sha256("../models/credit_scoring_pipeline.pkl"),
}, name="sha256")
hashes.to_csv("../reports/artifact_hashes.csv", index_label="artifact")
hashes

data/application_train.csv            52e96b895b1112e1c853f670e58372719c8441c5ed1c57...
models/credit_scoring_pipeline.pkl    81789d80e479dfd85a356a609de12ca946997dd6624486...
Name: sha256, dtype: object